# Lab 3.2 — Missing data with pandas

**Goals:** detect missing values, distinguish NaN from placeholder text, choose **drop** vs **impute** (`fillna`, `ffill`, `interpolate`), and see how choices affect the analysis.

Run **Setup** first. Complete each **Exercise** in the empty code cells. See the [pandas missing data guide](https://pandas.pydata.org/docs/user_guide/missing_data.html).

## Setup — imports and sample dataset

The table `df_raw` simulates daily sensor readings. Some gaps are real `NaN`, some cells are empty strings, and some use the sentinel text `N/A` that you should normalize before analysis.

In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 20)

rng = np.random.default_rng(7)
days = pd.date_range("2024-06-01", periods=18, freq="D")

base_temp = 18 + 6 * np.sin(np.linspace(0, 2 * np.pi, len(days))) + rng.normal(0, 0.8, len(days))
base_hum = 55 + 15 * np.cos(np.linspace(0, 2 * np.pi, len(days))) + rng.normal(0, 3, len(days))

df_raw = pd.DataFrame({
    "date": days,
    "sensor_id": rng.choice(["S1", "S2"], size=len(days)),
    "temp_c": np.round(base_temp, 1),
    "humidity_pct": np.clip(np.round(base_hum, 0), 20, 95),
    "qc_flag": rng.choice(["ok", "suspect", ""], size=len(days), p=[0.6, 0.25, 0.15]),
})

# inject various missing styles
for idx in [2, 5, 8, 12, 15]:
    df_raw.loc[idx, "temp_c"] = np.nan
for idx in [3, 9, 14]:
    df_raw.loc[idx, "humidity_pct"] = np.nan
df_raw.loc[4, "qc_flag"] = "N/A"
df_raw.loc[11, "qc_flag"] = ""
df_raw.loc[16, "qc_flag"] = "na"

df_raw

,date,sensor_id,temp_c,humidity_pct,qc_flag
0,2024-06-01,S1,18.0,64.0,ok
1,2024-06-02,S1,20.4,65.0,ok
2,2024-06-03,S2,NaN,61.0,ok
3,2024-06-04,S2,22.7,NaN,
4,2024-06-05,S2,23.6,53.0,N/A
5,2024-06-06,S1,NaN,52.0,ok
6,2024-06-07,S2,22.8,46.0,ok
7,2024-06-08,S1,22.2,42.0,ok
8,2024-06-09,S2,NaN,33.0,ok
9,2024-06-10,S2,16.4,NaN,ok


---
## Exercise 1 — Count missing values (as pandas sees them)

Use `isna()` (alias `isnull()`): **per-column** NaN counts for `df_raw`.

**Deliverable:** table/Series column → count. In short: which column looks clean in `isna()` but is actually dirty in real life?

In [4]:
# Exercise 1 — your code here



## Exercise 2 — Normalize text sentinels to missing

Copy `df_raw` to `df_clean`. In `qc_flag`, treat **empty string**, `N/A`, and `na` (any case) as missing (`np.nan`). Keep `ok` and `suspect`.

**Deliverable:** `value_counts(dropna=False)` for `qc_flag` after cleaning.

In [5]:
# Exercise 2 — your code here



## Exercise 3 — Filter incomplete numeric rows

`incomplete` = rows where `temp_c` **or** `humidity_pct` is NaN (use `df_clean`).

**Deliverable:** show `incomplete.shape`.

In [ ]:
# Exercise 3 — your code here



## Exercise 4 — Drop strategy

`df_drop`: from `df_clean`, **drop** rows with any missing in `temp_c` or `humidity_pct` (`subset` + `dropna`).

**Deliverable:** `len(df_drop)` and assert those two columns have zero NaNs.

In [ ]:
# Exercise 4 — your code here



## Exercise 5 — Impute strategy

From `df_clean`, build `df_fill`:

1. Sort by `date`.
2. `temp_c`: **forward-fill** along time (`ffill`).
3. `humidity_pct`: fill remaining NaNs with the **median** of `humidity_pct` from the original `df_clean` (or from non-NaN before fill — be consistent and state what you used).

**Deliverable:** `df_fill[["date", "temp_c", "humidity_pct"]].isna().sum()` should be 0 for `temp_c` and `humidity_pct`.

In [ ]:
# Exercise 5 — your code here



## Exercise 6 — Optional: time interpolate

On `df_clean` sorted by `date`, set `date` as index and run `interpolate(method="time")` on `temp_c` only (or on a one-column frame).

**Deliverable:** for one index where `temp_c` was missing, compare the value from **Exercise 5 ffill** vs **interpolate**. Are they the same? Why?

In [ ]:
# Exercise 6 — optional



---
## Reference solutions

Use only after attempting the exercises; instructors may delete this section.

In [ ]:
# Solutions (uncomment to run)
# ex1 = df_raw.isna().sum()
# display(ex1)

# df_clean = df_raw.copy()
# s = df_clean["qc_flag"].astype(str).str.strip().str.lower()
# df_clean.loc[s.isin(["", "n/a", "na"]), "qc_flag"] = np.nan
# display(df_clean["qc_flag"].value_counts(dropna=False))

# m = df_clean["temp_c"].isna() | df_clean["humidity_pct"].isna()
# incomplete = df_clean.loc[m]
# print(incomplete.shape)

# df_drop = df_clean.dropna(subset=["temp_c", "humidity_pct"], how="any")
# assert df_drop[["temp_c", "humidity_pct"]].notna().all().all()

# df_fill = df_clean.sort_values("date").copy()
# df_fill["temp_c"] = df_fill["temp_c"].ffill()
# med = df_clean["humidity_pct"].median()
# df_fill["humidity_pct"] = df_fill["humidity_pct"].fillna(med)
# display(df_fill[["date", "temp_c", "humidity_pct"]].isna().sum())

# t_interp = (
#     df_clean.sort_values("date")
#     .set_index("date")["temp_c"]
#     .interpolate(method="time")
# )
# df_fill_i = df_clean.sort_values("date").copy()
# df_fill_i["temp_c"] = df_fill_i["temp_c"].ffill()
# for d in df_clean.loc[df_clean["temp_c"].isna(), "date"]:
#     print(d.date(), "ffill", df_fill_i.set_index("date").loc[d, "temp_c"], "interp", t_interp.loc[d])